# Визуализация предсказаний YOLOv9c на val dataset2_split_90_10

**GT** (зелёный) vs **Pred** (красный) с confidence.

Модель: `dataset2_90_10_yolov9c_newhyp/weights/best.pt` (epoch 26)

In [7]:
import os
import random
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# Paths
IMG_DIR  = "data/dataset2_split_90_10/valid/images"
GT_DIR   = "data/dataset2_split_90_10/valid/labels"
PRED_DIR = "yolo/yolov9/runs/train/dataset2_90_10_yolov9c_newhyp/val_90_102/labels"

all_images = sorted(os.listdir(IMG_DIR))
print(f"Total val images: {len(all_images)}")

Total val images: 833


In [8]:
def parse_gt(label_path):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls = int(parts[0])
                cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                boxes.append((cls, cx, cy, w, h))
    return boxes


def parse_pred(label_path):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 6:
                cls = int(parts[0])
                cx, cy, w, h, conf = (float(parts[1]), float(parts[2]),
                                      float(parts[3]), float(parts[4]), float(parts[5]))
                boxes.append((cls, cx, cy, w, h, conf))
    return boxes


def draw_yolo_box(ax, cx, cy, w, h, img_w, img_h, color, label, linewidth=2):
    x1 = (cx - w / 2) * img_w
    y1 = (cy - h / 2) * img_h
    bw = w * img_w
    bh = h * img_h
    rect = patches.Rectangle((x1, y1), bw, bh, linewidth=linewidth,
                             edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1 - 2, label, color=color, fontsize=7, fontweight='bold',
            bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', pad=0.5))


def compute_iou_xyxy(b1, b2):
    x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    return inter / (a1 + a2 - inter) if (a1 + a2 - inter) > 0 else 0

## Статистика на валидации

In [9]:
CONF_THRESH = 0.1
IOU_MATCH = 0.3

total_tp = total_fp = total_fn = 0
results_per_image = []

for img_name in all_images:
    stem = os.path.splitext(img_name)[0]
    img = cv2.imread(os.path.join(IMG_DIR, img_name))
    img_h, img_w = img.shape[:2]

    gt_boxes = parse_gt(os.path.join(GT_DIR, stem + ".txt"))
    pred_boxes = [p for p in parse_pred(os.path.join(PRED_DIR, stem + ".txt")) if p[5] >= CONF_THRESH]

    def to_xyxy(cx, cy, w, h):
        return ((cx - w/2)*img_w, (cy - h/2)*img_h, (cx + w/2)*img_w, (cy + h/2)*img_h)

    gt_xyxy = [to_xyxy(cx, cy, w, h) for _, cx, cy, w, h in gt_boxes]
    pred_xyxy = [to_xyxy(cx, cy, w, h) for _, cx, cy, w, h, _ in pred_boxes]

    matched_gt = set()
    tp = 0
    for pb in pred_xyxy:
        best_iou, best_gi = 0, -1
        for gi, gb in enumerate(gt_xyxy):
            if gi in matched_gt:
                continue
            iou = compute_iou_xyxy(pb, gb)
            if iou > best_iou:
                best_iou, best_gi = iou, gi
        if best_iou >= IOU_MATCH:
            matched_gt.add(best_gi)
            tp += 1

    fp = len(pred_xyxy) - tp
    fn = len(gt_xyxy) - len(matched_gt)
    total_tp += tp
    total_fp += fp
    total_fn += fn
    results_per_image.append((img_name, len(gt_boxes), len(pred_boxes), tp, fp, fn))

prec = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
rec  = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0

print(f"Conf threshold: {CONF_THRESH}  |  IoU match: {IOU_MATCH}")
print(f"TP: {total_tp}  FP: {total_fp}  FN: {total_fn}")
print(f"Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
print(f"Images with GT but no preds: {sum(1 for _, ngt, npr, *_ in results_per_image if ngt > 0 and npr == 0)}")

Conf threshold: 0.1  |  IoU match: 0.3
TP: 244  FP: 506  FN: 589
Precision: 0.3253  Recall: 0.2929  F1: 0.3083
Images with GT but no preds: 368


## Рандомные 100 изображений — GT (green) vs Pred (red)

In [10]:
random.seed(42)
sampled = random.sample(all_images, min(100, len(all_images)))

NCOLS = 4
NROWS = 25
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(20, 100))
axes = axes.flatten()

for idx, img_name in enumerate(sampled):
    stem = os.path.splitext(img_name)[0]
    img_path = os.path.join(IMG_DIR, img_name)
    gt_path = os.path.join(GT_DIR, stem + ".txt")
    pred_path = os.path.join(PRED_DIR, stem + ".txt")

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_h, img_w = img.shape[:2]

    gt_boxes = parse_gt(gt_path)
    pred_boxes = parse_pred(pred_path)

    ax = axes[idx]
    ax.imshow(img)
    ax.set_title(stem[:40], fontsize=6)
    ax.axis('off')

    for cls, cx, cy, w, h in gt_boxes:
        draw_yolo_box(ax, cx, cy, w, h, img_w, img_h, 'lime', 'GT', linewidth=2)

    for cls, cx, cy, w, h, conf in pred_boxes:
        if conf >= CONF_THRESH:
            draw_yolo_box(ax, cx, cy, w, h, img_w, img_h, 'red', f'{conf:.2f}', linewidth=1)

for idx in range(len(sampled), len(axes)):
    axes[idx].axis('off')

plt.suptitle(f"dataset2_90_10 val — GT (green) vs Pred (red)  conf>={CONF_THRESH}",
             fontsize=16, y=1.001)
plt.tight_layout()
plt.show()

KeyboardInterrupt: 

## Лучшие предсказания (TP — модель попала)

In [ ]:
# Images with highest TP count
best_tp = sorted(results_per_image, key=lambda x: x[3], reverse=True)[:16]

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes = axes.flatten()

for idx, (img_name, ngt, npred, tp, fp, fn) in enumerate(best_tp):
    stem = os.path.splitext(img_name)[0]
    img = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, img_name)), cv2.COLOR_BGR2RGB)
    img_h, img_w = img.shape[:2]

    gt_boxes = parse_gt(os.path.join(GT_DIR, stem + ".txt"))
    pred_boxes = [p for p in parse_pred(os.path.join(PRED_DIR, stem + ".txt")) if p[5] >= CONF_THRESH]

    ax = axes[idx]
    ax.imshow(img)
    ax.set_title(f"{stem[:30]}  TP={tp} FP={fp} FN={fn}", fontsize=7)
    ax.axis('off')

    for cls, cx, cy, w, h in gt_boxes:
        draw_yolo_box(ax, cx, cy, w, h, img_w, img_h, 'lime', 'GT', linewidth=2)
    for cls, cx, cy, w, h, conf in pred_boxes:
        draw_yolo_box(ax, cx, cy, w, h, img_w, img_h, 'red', f'{conf:.2f}', linewidth=1)

plt.suptitle("Best predictions (most TP)", fontsize=14)
plt.tight_layout()
plt.show()

## Худшие предсказания (FN — модель пропустила)

In [ ]:
# Images with highest FN count (most missed GT)
worst_fn = sorted(results_per_image, key=lambda x: x[5], reverse=True)[:16]

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes = axes.flatten()

for idx, (img_name, ngt, npred, tp, fp, fn) in enumerate(worst_fn):
    stem = os.path.splitext(img_name)[0]
    img = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, img_name)), cv2.COLOR_BGR2RGB)
    img_h, img_w = img.shape[:2]

    gt_boxes = parse_gt(os.path.join(GT_DIR, stem + ".txt"))
    pred_boxes = [p for p in parse_pred(os.path.join(PRED_DIR, stem + ".txt")) if p[5] >= CONF_THRESH]

    ax = axes[idx]
    ax.imshow(img)
    ax.set_title(f"{stem[:30]}  TP={tp} FP={fp} FN={fn}", fontsize=7)
    ax.axis('off')

    for cls, cx, cy, w, h in gt_boxes:
        draw_yolo_box(ax, cx, cy, w, h, img_w, img_h, 'lime', 'GT', linewidth=2)
    for cls, cx, cy, w, h, conf in pred_boxes:
        draw_yolo_box(ax, cx, cy, w, h, img_w, img_h, 'red', f'{conf:.2f}', linewidth=1)

plt.suptitle("Worst misses (most FN)", fontsize=14)
plt.tight_layout()
plt.show()

## False Positives — модель предсказала, но GT нет

In [ ]:
# Images with highest FP count
worst_fp = sorted(results_per_image, key=lambda x: x[4], reverse=True)[:16]

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
axes = axes.flatten()

for idx, (img_name, ngt, npred, tp, fp, fn) in enumerate(worst_fp):
    stem = os.path.splitext(img_name)[0]
    img = cv2.cvtColor(cv2.imread(os.path.join(IMG_DIR, img_name)), cv2.COLOR_BGR2RGB)
    img_h, img_w = img.shape[:2]

    gt_boxes = parse_gt(os.path.join(GT_DIR, stem + ".txt"))
    pred_boxes = [p for p in parse_pred(os.path.join(PRED_DIR, stem + ".txt")) if p[5] >= CONF_THRESH]

    ax = axes[idx]
    ax.imshow(img)
    ax.set_title(f"{stem[:30]}  TP={tp} FP={fp} FN={fn}", fontsize=7)
    ax.axis('off')

    for cls, cx, cy, w, h in gt_boxes:
        draw_yolo_box(ax, cx, cy, w, h, img_w, img_h, 'lime', 'GT', linewidth=2)
    for cls, cx, cy, w, h, conf in pred_boxes:
        draw_yolo_box(ax, cx, cy, w, h, img_w, img_h, 'red', f'{conf:.2f}', linewidth=1)

plt.suptitle("Worst false positives (most FP)", fontsize=14)
plt.tight_layout()
plt.show()

## Распределение confidence предсказаний

In [ ]:
all_confs = []
for img_name in all_images:
    stem = os.path.splitext(img_name)[0]
    preds = parse_pred(os.path.join(PRED_DIR, stem + ".txt"))
    all_confs.extend([p[5] for p in preds])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(all_confs, bins=50, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Confidence')
ax1.set_ylabel('Count')
ax1.set_title(f'All predictions confidence distribution (n={len(all_confs)})')
ax1.axvline(x=CONF_THRESH, color='red', linestyle='--', label=f'threshold={CONF_THRESH}')
ax1.legend()

confs_above = [c for c in all_confs if c >= CONF_THRESH]
ax2.hist(confs_above, bins=30, edgecolor='black', alpha=0.7, color='orange')
ax2.set_xlabel('Confidence')
ax2.set_ylabel('Count')
ax2.set_title(f'Predictions above threshold (n={len(confs_above)})')

plt.tight_layout()
plt.show()

print(f"Total predictions: {len(all_confs)}")
print(f"Above conf {CONF_THRESH}: {len(confs_above)}")
print(f"Max conf: {max(all_confs):.4f}" if all_confs else "No predictions")